# Pig Posture Recognition - V8B Inference

**Modell:** DINOv2 Large @ 518px (Student, trainiert mit Pseudo-Labels + KD aus V5+V6-Teacher).

Identisch zu V8a-Pipeline, nur anderer CKPT_DIR. Das KD-trainierte Modell sollte
etwas besser generalisieren als V8a (erwartet +0.3-0.8% F1).

## Configuration

In [1]:
import os

TAG = "T2"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

_candidates = [
    "multiview_pig_posture_recognition",
    "./multiview_pig_posture_recognition",
    "/datasets/multi-view-pig-posture-recognition",
    "/multi-view-pig-posture-recognition",
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None
print(f"DATA_ROOT = {os.path.abspath(DATA_ROOT)}")

TEST_CSV  = os.path.join(DATA_ROOT, "test.csv")
IMG_DIR   = os.path.join(DATA_ROOT, "test_images")

CKPT_DIR  = f"runs/v8b_{TAG.lower()}"
CKPT_PATHS = sorted([
    os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)
    if f.startswith("best_dinov2l_fold_") and f.endswith(".pth")
])

OUTPUT_FILE = f"{TAG}_v8b_dinov2_kd.csv"

INFER_IMG_SIZE = {"dinov2l": 518}

BATCH_SIZE   = 32
NUM_WORKERS  = 16
USE_TTA      = True
PAD_RATIO    = 0.1
NUM_CLASSES  = 5

CLASS_NAMES = ["Lateral_lying_left", "Lateral_lying_right",
               "Sitting", "Standing", "Sternal_lying"]

ADAPT_BN         = False
BN_ADAPT_BATCHES = 50

EXPORT_PSEUDO_LABELS   = True
PSEUDO_TOP_K_PER_CLASS = 200
PSEUDO_MIN_CONFIDENCE  = 0.60
PSEUDO_OUTPUT          = f"pseudo_labels_{TAG.lower()}_v8b.csv"

print(f"Tag: {TAG}  |  Version: v8b  |  Checkpoints: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {os.path.basename(p)}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  Version: v8b  |  Checkpoints: 3
  - best_dinov2l_fold_1.pth
  - best_dinov2l_fold_2.pth
  - best_dinov2l_fold_3.pth


## Imports

In [2]:
import os, ast, re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

<jemalloc>: Unsupported system page size


Device: cuda


## Test Dataset + TTA

In [3]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.1):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform:
            crop = self.transform(crop)
        return crop, row["row_id"]


NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

def build_tta_configs(img_size):
    S = img_size
    return [
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                    T.ToTensor(), T.Normalize(*NORM)]), True),
        (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                    T.CenterCrop(S),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
        (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                    T.ToTensor(), T.Normalize(*NORM)]), False),
    ]

## ViT Loader + TTA Helper

In [4]:
def infer_prefix(ckpt_path):
    name = os.path.basename(ckpt_path)
    m = re.match(r"best_([a-zA-Z0-9]+)_fold_\d+\.pth", name)
    return m.group(1) if m else "unknown"


def load_vit_with_resampling(name, ckpt, num_classes, infer_size):
    model = timm.create_model(name, pretrained=False, num_classes=num_classes, img_size=infer_size)
    state_dict = dict(ckpt["model"])

    if "pos_embed" in state_dict:
        old_pe = state_dict["pos_embed"]
        new_pe = model.pos_embed
        if old_pe.shape != new_pe.shape:
            try:
                from timm.layers import resample_abs_pos_embed
            except ImportError:
                from timm.models.layers import resample_abs_pos_embed
            new_grid = model.patch_embed.grid_size
            num_prefix = getattr(model, "num_prefix_tokens", 1)
            print(f"    Interpoliere pos_embed: {tuple(old_pe.shape)} -> {tuple(new_pe.shape)} "
                  f"(grid={new_grid}, prefix_tokens={num_prefix})")
            state_dict["pos_embed"] = resample_abs_pos_embed(
                old_pe, new_size=new_grid, num_prefix_tokens=num_prefix
            )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"    Missing keys: {missing[:3]}{'...' if len(missing)>3 else ''}")
    if unexpected:
        print(f"    Unexpected keys: {unexpected[:3]}{'...' if len(unexpected)>3 else ''}")
    return model


@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs, batch_size):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f"  TTA {i+1}/{len(tta_configs)}", leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)

print("Helpers geladen.")

Helpers geladen.


## Inferenz ueber alle Folds

In [5]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")

all_probs = []

for idx, path in enumerate(CKPT_PATHS):
    prefix = infer_prefix(path)
    print(f"\nModell {idx+1}/{len(CKPT_PATHS)}: {os.path.basename(path)}")

    ckpt = torch.load(path, map_location="cpu")
    name = ckpt.get("model_name", "unknown")
    ckpt_img_size = ckpt.get("img_size", 518)
    infer_size = INFER_IMG_SIZE.get(prefix, ckpt_img_size)

    print(f"  {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  "
          f"KD: alpha={ckpt.get('kd_alpha','?')}, T={ckpt.get('kd_T','?')}  |  "
          f"Infer@{infer_size}px")

    model = load_vit_with_resampling(name, ckpt, NUM_CLASSES, infer_size)
    model.to(DEVICE).eval()

    tta_cfg = build_tta_configs(infer_size) if USE_TTA else [build_tta_configs(infer_size)[0]]
    fold_probs = predict_tta(model, test_df, IMG_DIR, tta_cfg, BATCH_SIZE)
    all_probs.append(fold_probs)

    del model
    import gc; gc.collect()
    torch.cuda.empty_cache()

final_probs = np.mean(all_probs, axis=0)
predictions = final_probs.argmax(axis=1)

print(f"\n{len(predictions)} Vorhersagen aus {len(CKPT_PATHS)} Folds")

Test-Instanzen: 11708

Modell 1/3: best_dinov2l_fold_1.pth
  vit_large_patch14_dinov2.lvd142m  |  Val F1: 0.8160  |  KD: alpha=0.5, T=4.0  |  Infer@518px


  TTA 1/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

...

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/366 [00:00<?, ?it/s]


11708 Vorhersagen aus 3 Folds


## Submission

In [6]:
submission = pd.DataFrame({
    "row_id": test_df["row_id"].values,
    "class_id": predictions.astype(int)
})
submission.to_csv(OUTPUT_FILE, index=False)

assert list(submission.columns) == ["row_id", "class_id"]
assert set(submission["class_id"].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)

print(f"Submission: {OUTPUT_FILE} ({len(submission)} Zeilen)\n")
print(f"Verteilung:")
for c in range(NUM_CLASSES):
    cnt = (submission["class_id"] == c).sum()
    pct = 100 * cnt / len(submission)
    bar = "#" * int(30 * cnt / len(submission))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

submission.head(10)

Submission: T2_v8b_dinov2_kd.csv (11708 Zeilen)

Verteilung:
  0 - Lateral_lying_left     ###                             1339 (11.4%)
  1 - Lateral_lying_right    ###                             1544 (13.2%)
  2 - Sitting                #                                547 (4.7%)
  3 - Standing               ##############                  5812 (49.6%)
  4 - Sternal_lying          ######                          2466 (21.1%)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,1
2,test_pen1_tur_cam1_20250920_174649_0002,1
3,test_pen1_tur_cam1_20250920_174649_0003,1
4,test_pen1_tur_cam1_20250920_174649_0004,0
5,test_pen1_tur_cam1_20250920_174649_0005,1
6,test_pen1_tur_cam1_20250920_174649_0006,1
7,test_pen1_tur_cam1_20250920_174649_0007,1
8,test_pen1_tur_cam1_20250920_174649_0008,0
9,test_pen1_tur_cam1_20250921_050022_0000,0


## Optional: Pseudo-Labels fuer naechste Iteration

In [7]:
if EXPORT_PSEUDO_LABELS:
    max_probs = final_probs.max(axis=1)
    argmax = final_probs.argmax(axis=1)

    selected_idx = []
    print(f"Class-balanced Top-{PSEUDO_TOP_K_PER_CLASS} pro Klasse "
          f"(min conf {PSEUDO_MIN_CONFIDENCE}):")
    for c in range(NUM_CLASSES):
        cand_mask = (argmax == c) & (max_probs >= PSEUDO_MIN_CONFIDENCE)
        cand_idx = np.where(cand_mask)[0]
        cand_idx = cand_idx[np.argsort(-max_probs[cand_idx])]
        chosen = cand_idx[:PSEUDO_TOP_K_PER_CLASS]
        selected_idx.extend(chosen.tolist())
        if len(chosen) > 0:
            print(f"  {c} - {CLASS_NAMES[c]:<22} "
                  f"{len(chosen):>4} ausgewaehlt "
                  f"(min conf={max_probs[chosen].min():.3f}, "
                  f"max conf={max_probs[chosen].max():.3f})")
        else:
            print(f"  {c} - {CLASS_NAMES[c]:<22} KEINE Kandidaten")
    selected_idx = np.array(sorted(selected_idx), dtype=int)
    mask = np.zeros(len(test_df), dtype=bool)
    mask[selected_idx] = True

    pseudo_df = test_df[mask].copy()
    pseudo_df["class_id"] = predictions[mask].astype(int)
    pseudo_df["confidence"] = max_probs[mask]
    pseudo_df.to_csv(PSEUDO_OUTPUT, index=False)

    n = mask.sum()
    print(f"\nPseudo-Labels: {PSEUDO_OUTPUT}  ({n}/{len(test_df)})")
else:
    print("Pseudo-Label Export deaktiviert.")

Class-balanced Top-200 pro Klasse (min conf 0.6):
  0 - Lateral_lying_left      200 ausgewaehlt (min conf=0.948, max conf=0.953)
  1 - Lateral_lying_right     200 ausgewaehlt (min conf=0.946, max conf=0.949)
  2 - Sitting                 200 ausgewaehlt (min conf=0.993, max conf=0.995)
  3 - Standing                200 ausgewaehlt (min conf=0.844, max conf=0.848)
  4 - Sternal_lying           200 ausgewaehlt (min conf=0.883, max conf=0.896)

Pseudo-Labels: pseudo_labels_t2_v8b.csv  (1000/11708)
